In [72]:
import csv
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tabulate import tabulate


In [75]:

URL = "https://books.toscrape.com/"
API_URL = "https://exchangerate-api.com"

In [76]:
response = requests.get(URL)
response

<Response [200]>

In [82]:
def fetch_exchange_rate():
    """Fetches the current GBP to USD exchange rate with error handling."""
    try:
        response = requests.get(API_URL, timeout=10)
        response.raise_for_status()
        data = response.json()
        # API returns rates against GBP
        return data['rates'].get('USD', 1.0) 
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to currency API: {e}")
        print("Defaulting to an exchange rate of 1.0")
        return 1.0


In [80]:
def scrape_books():
    """Scrapes book titles and prices from the target website."""
    try:
        response = requests.get(URL, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to website: {e}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    books = soup.find_all('article', class_='product_pod')
    
    scraped_data = []
    for book in books:
        # Clean and extract title
        title = book.h3.a['title']
        
        # Clean and extract price
        price_str = book.find('p', class_='price_color').text
        # Remove currency symbol (Â£) and convert to float
        price_float = float(price_str.replace('Â£', ''))
        
        scraped_data.append({
            'Book Title': title,
            'Original Price (GBP)': price_float
        })
        
        # Stop at 10 products as requested
        if len(scraped_data) >= 10:
            break
            
    return scraped_data

In [81]:
def main():
    print("Scraping website...")
    books = scrape_books()
    
    if not books:
        print("No data collected. Exiting.")
        return

    print("Fetching exchange rates...")
    exchange_rate = fetch_exchange_rate()

    # Calculate converted price
    for book in books:
        original_price = book['Original Price (GBP)']
        converted_price = original_price * exchange_rate
        book['Converted Price (USD)'] = round(converted_price, 2)

    # Convert to pandas DataFrame for easy table display
    df = pd.DataFrame(books)

    # Display in a readable table
    print("\nProduct Price Comparison:")
    print(tabulate(df, headers='keys', tablefmt='psql', showindex=False))

    # Save to CSV
    output_filename = 'books_price_conversion.csv'
    df.to_csv(output_filename, index=False)
    print(f"\nData successfully saved to {output_filename}")

if __name__ == "__main__":
    main()

Scraping website...
Fetching exchange rates...
Error connecting to currency API: Expecting value: line 1 column 1 (char 0)
Defaulting to an exchange rate of 1.0

Product Price Comparison:
+------------------------------------------------------------------------------------------------+------------------------+-------------------------+
| Book Title                                                                                     |   Original Price (GBP) |   Converted Price (USD) |
|------------------------------------------------------------------------------------------------+------------------------+-------------------------|
| A Light in the Attic                                                                           |                  51.77 |                   51.77 |
| Tipping the Velvet                                                                             |                  53.74 |                   53.74 |
| Soumission                                                  

PermissionError: [Errno 13] Permission denied: 'books_price_conversion.csv'